In [3]:
import sys
import pickle
import numpy as np

!{sys.executable} -m pip install sentence-transformers scikit-learn numpy

from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

sentences = [
    "Virat Kohli scored a century in the cricket match.",
    "The bowler delivered a fast yorker to take the wicket.",
    "Cricket fans celebrated the thrilling last over finish.",
    "The team practiced batting and fielding before the tournament.",

    "Cooking pasta requires boiling water and preparing sauce.",
    "The chef added spices to improve the flavor of the curry.",
    "Baking cakes becomes easier with accurate measurements.",

    "Python is widely used for machine learning projects.",
    "Functions and loops are basic programming concepts.",
    "Debugging code helps programmers identify software errors."
]

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(sentences)

nn_model = NearestNeighbors(
    n_neighbors=2,
    metric="cosine"
)

nn_model.fit(embeddings)

queries = [
    "Best cricket batting performance",
    "How to cook delicious food",
    "Programming and debugging code"
]

print("\nSearch Results Before Saving:\n")

before_results = []

for query in queries:

    query_embedding = model.encode([query])

    distances, indices = nn_model.kneighbors(query_embedding)

    query_result = []

    print(f"\nQuery: {query}\n")

    for rank, idx in enumerate(indices[0]):

        result = {
            "sentence": sentences[idx],
            "distance": float(distances[0][rank])
        }

        query_result.append(result)

        print(f"Top {rank+1}:")
        print("Sentence:", sentences[idx])
        print("Distance:", distances[0][rank])
        print()

    before_results.append(query_result)

with open("vector_index.pkl", "wb") as file:
    pickle.dump(nn_model, file)

with open("documents.pkl", "wb") as file:
    pickle.dump(sentences, file)

with open("vector_index.pkl", "rb") as file:
    loaded_model = pickle.load(file)

with open("documents.pkl", "rb") as file:
    loaded_sentences = pickle.load(file)

print("\nSearch Results After Reload:\n")

after_results = []

for query in queries:

    query_embedding = model.encode([query])

    distances, indices = loaded_model.kneighbors(query_embedding)

    query_result = []

    print(f"\nQuery: {query}\n")

    for rank, idx in enumerate(indices[0]):

        result = {
            "sentence": loaded_sentences[idx],
            "distance": float(distances[0][rank])
        }

        query_result.append(result)

        print(f"Top {rank+1}:")
        print("Sentence:", loaded_sentences[idx])
        print("Distance:", distances[0][rank])
        print()

    after_results.append(query_result)

if before_results == after_results:
    print("\nVerification Successful:")
    print("Search results are identical after reload.")
else:
    print("\nVerification Failed:")
    print("Search results changed after reload.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Search Results Before Saving:


Query: Best cricket batting performance

Top 1:
Sentence: The team practiced batting and fielding before the tournament.
Distance: 0.4784943

Top 2:
Sentence: Virat Kohli scored a century in the cricket match.
Distance: 0.48314238


Query: How to cook delicious food

Top 1:
Sentence: The chef added spices to improve the flavor of the curry.
Distance: 0.57371163

Top 2:
Sentence: Cooking pasta requires boiling water and preparing sauce.
Distance: 0.5897609


Query: Programming and debugging code

Top 1:
Sentence: Debugging code helps programmers identify software errors.
Distance: 0.21893716

Top 2:
Sentence: Functions and loops are basic programming concepts.
Distance: 0.4748881


Search Results After Reload:


Query: Best cricket batting performance

Top 1:
Sentence: The team practiced batting and fielding before the tournament.
Distance: 0.4784943

Top 2:
Sentence: Virat Kohli scored a century in the cricket match.
Distance: 0.48314238


Query: How to